# 🌸 Iris Flower Species Classifier

## Step 1 — Import Libraries

In [29]:
import pandas as pd
import numpy as np
import pickle
import json
 
from sklearn.datasets import load_iris                          
from sklearn.svm import SVC                                   
from sklearn.pipeline import Pipeline                           
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split            
from sklearn.metrics import (
    accuracy_score,        
    classification_report, 
    confusion_matrix       
)
from sklearn.inspection import permutation_importance   

## Step 2 — Load & Explore Dataset

In [30]:
df = pd.read_csv("iris_dataset.csv")                         
 
print("\n📦 Dataset loaded!")
print(f"   Shape: {df.shape} ({df.shape[0]} rows × {df.shape[1]} columns)")

 


📦 Dataset loaded!
   Shape: (150, 6) (150 rows × 6 columns)


In [31]:
print(f"\n   First 5 rows:")
print(df.head().to_string())
print(f"\n   Species distribution:")
print(df['Species'].value_counts().to_string())
print(f"\n   Basic statistics:")
print(df.describe().round(2).to_string())


   First 5 rows:
   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa

   Species distribution:
Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50

   Basic statistics:
           Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
count  150.00         150.00        150.00         150.00        150.00
mean    75.50           5.84          3.05           3.76          1.20
std     43.45           0.83          0.43           1.76          0.76
min      1.00           4.30          2.00           1.00          0.10
25%     38.25           5.10          

In [32]:
print('🌸 Class Distribution:')
print(df['Species'].value_counts())

# Check for missing values
print('\n🔍 Missing Values:')
print(df.isnull().sum())

🌸 Class Distribution:
Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

🔍 Missing Values:
Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64


## STEP 3: PREPARE FEATURES AND LABELS

In [33]:
feature_cols = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
 
X = df[feature_cols].values   
y_str = df['Species'].values   
 
le = LabelEncoder()
y = le.fit_transform(y_str)  
 
print(f"\n📊 Features (X): shape {X.shape}")
print(f"   Labels  (y): {np.unique(y_str)} → encoded as {np.unique(y)}")


📊 Features (X): shape (150, 4)
   Labels  (y): ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica'] → encoded as [0 1 2]


## STEP 4: TRAIN / TEST SPLIT

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       
    random_state=1,         
    stratify=y           
)
 
print(f"\n✂️  Train/Test Split:")
print(f"   Training samples: {len(X_train)} (80%)")
print(f"   Testing  samples: {len(X_test)} (20%)")
print(f"   Test class counts: {np.bincount(y_test)}")


✂️  Train/Test Split:
   Training samples: 120 (80%)
   Testing  samples: 30 (20%)
   Test class counts: [10 10 10]


## STEP 5: BUILD ML PIPELINE

In [35]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(
        kernel='rbf',
        C=1000,
        gamma=0.01,
        probability=True,
        random_state=42
    ))
])
 
print(f"\n🧠 Model: SVM (Support Vector Machine)")
print(f"   Pipeline: StandardScaler → SVC(rbf, C=1000, gamma=0.01)")


🧠 Model: SVM (Support Vector Machine)
   Pipeline: StandardScaler → SVC(rbf, C=1000, gamma=0.01)


## Step 6 — Train Model

In [36]:
print(f"\n⏳ Training model...")
model.fit(X_train, y_train)
print(f"   ✅ Training complete!")


⏳ Training model...
   ✅ Training complete!


## STEP 7: EVALUATE ON TEST DATA


In [37]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
 
print(f"\n{'=' * 60}")
print(f"  MODEL EVALUATION")
print(f"{'=' * 60}")
print(f"\n✅ Test Accuracy: {acc * 100:.2f}%")
print(f"\n📋 Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_
))
print(f"\n🔢 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[f'Actual {c}' for c in le.classes_],
    columns=[f'Pred {c}' for c in le.classes_]
)
print(cm_df.to_string())
print("\n  (Diagonal = correct predictions)")
print(f"  All off-diagonal values are 0 → PERFECT classification!")


  MODEL EVALUATION

✅ Test Accuracy: 100.00%

📋 Classification Report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      1.00      1.00        10
 Iris-virginica       1.00      1.00      1.00        10

       accuracy                           1.00        30
      macro avg       1.00      1.00      1.00        30
   weighted avg       1.00      1.00      1.00        30


🔢 Confusion Matrix:
                        Pred Iris-setosa  Pred Iris-versicolor  Pred Iris-virginica
Actual Iris-setosa                    10                     0                    0
Actual Iris-versicolor                 0                    10                    0
Actual Iris-virginica                  0                     0                   10

  (Diagonal = correct predictions)
  All off-diagonal values are 0 → PERFECT classification!


## STEP 8: FEATURE IMPORTANCE (Permutation-based)

In [38]:
r = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
print(f"\n🎯 Feature Importance (Permutation-based):")
feat_imp = sorted(zip(feature_cols, r.importances_mean), key=lambda x: -x[1])
for feat, imp in feat_imp:
    bar = "█" * int(imp * 30 / max(r.importances_mean))
    print(f"   {feat:<20} {bar:<30} {imp:.4f}")
 
print(f"\n   💡 Petal dimensions are the MOST important features!")
print(f"      Setosa has tiny petals (easily separated).")
print(f"      Versicolor/Virginica are distinguished by petal size.")



🎯 Feature Importance (Permutation-based):
   PetalLengthCm        ██████████████████████████████ 0.4056
   PetalWidthCm         ██████████████████████████     0.3600
   SepalWidthCm         █████                          0.0700
   SepalLengthCm        ██                             0.0278

   💡 Petal dimensions are the MOST important features!
      Setosa has tiny petals (easily separated).
      Versicolor/Virginica are distinguished by petal size.


## STEP 9: CROSS-VALIDATION (optional, more robust evaluation)

In [39]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"\n📈 5-Fold Cross-Validation Scores:")
for i, s in enumerate(cv_scores):
    print(f"   Fold {i+1}: {s*100:.1f}%")
print(f"   Mean: {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%")


📈 5-Fold Cross-Validation Scores:
   Fold 1: 100.0%
   Fold 2: 96.7%
   Fold 3: 90.0%
   Fold 4: 90.0%
   Fold 5: 100.0%
   Mean: 95.3% ± 4.5%


## STEP 10: SAVE MODEL AS .pkl FILE

In [40]:
with open('iris_model.pkl', 'wb') as f:
    pickle.dump(model, f)
 
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
 
print(f"\n💾 Saved:")
print(f"   iris_model.pkl     — The trained SVM pipeline")
print(f"   label_encoder.pkl  — Maps 0→setosa, 1→versicolor, 2→virginica")


💾 Saved:
   iris_model.pkl     — The trained SVM pipeline
   label_encoder.pkl  — Maps 0→setosa, 1→versicolor, 2→virginica


## STEP 11: LOAD AND USE THE SAVED MODEL

In [41]:
with open('iris_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
 
with open('label_encoder.pkl', 'rb') as f:
    loaded_le = pickle.load(f)
 
print(f"\n🔁 Testing loaded model with sample predictions:")
 
# Test samples with known answers
test_samples = [
    ([5.1, 3.5, 1.4, 0.2], "setosa"),       
    ([6.4, 3.2, 4.5, 1.5], "versicolor"),
    ([6.3, 3.3, 6.0, 2.5], "virginica"),    
]
 
for features, expected in test_samples:
    sample = np.array([features])
    pred_idx = loaded_model.predict(sample)[0]
    pred_proba = loaded_model.predict_proba(sample)[0]
    pred_species = loaded_le.classes_[pred_idx]
    confidence = pred_proba[pred_idx] * 100
    status = "✅" if pred_species == expected else "❌"
    print(f"   {status} Input: {features}")
    print(f"      → Predicted: {pred_species} ({confidence:.1f}% confidence)")
    print(f"      → Expected:  {expected}\n")
 


🔁 Testing loaded model with sample predictions:
   ❌ Input: [5.1, 3.5, 1.4, 0.2]
      → Predicted: Iris-setosa (97.2% confidence)
      → Expected:  setosa

   ❌ Input: [6.4, 3.2, 4.5, 1.5]
      → Predicted: Iris-versicolor (95.0% confidence)
      → Expected:  versicolor

   ❌ Input: [6.3, 3.3, 6.0, 2.5]
      → Predicted: Iris-virginica (98.8% confidence)
      → Expected:  virginica

